# 00 数据准备

从内嵌样本加载测试用例，自动生成反事实变体和掩码版本。

## 实验概述

**目的：** 为后续 notebook（01-06）准备标准化的测试数据，包括原始新闻、反事实变体和掩码版本。

**数据来源：** 42 条 A 股重大新闻事件（2020-2024），涵盖政策、企业、行业、宏观四大类别。

**两种反事实变体（用于泄露检测）：**
- `reverse_outcome`：翻转核心结论方向（利好↔利空）。非泄露模型应翻转预测，PC 高 = 泄露。
- `alter_numbers`：大幅修改数值至可能改变结论方向。非泄露模型应改变判断，PC 高 = 泄露。

**注意：** 实体替换（swap_entity）不作为反事实攻击变体。它测量的是实体名依赖性而非反事实鲁棒性，归入 notebook 03 的消融体系作为独立的实体替换实验。

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src import set_seed
from src.models import *
from src.llm_client import LLMClient
from src.news_loader import load_test_cases, save_counterfactual_variants
from src.masking import generate_counterfactuals_batch, apply_masking
from src.display_utils import show_comparison, show_llm_response
import json

set_seed()
(PROJECT_ROOT / "data" / "results").mkdir(parents=True, exist_ok=True)

## 1. 加载测试用例

In [2]:
test_cases = load_test_cases()
print(f"Loaded {len(test_cases)} test cases")
for tc in test_cases[:10]:
    print(f"  [{tc.id}] {tc.news.title} -> {tc.expected_direction.value}")

Loaded 42 test cases
  [tc_001] 央行宣布降准降息，推出多项金融支持政策 -> up
  [tc_002] A股三大指数大幅低开，沪指跌破2700点 -> down
  [tc_003] 中央政治局会议定调下半年经济工作 -> up
  [tc_004] 中办国办印发意见：进一步减轻义务教育阶段学生作业负担和校外培训负担 -> down
  [tc_005] 证监会主席吴清首次公开亮相，强调从严监管 -> neutral
  [tc_006] 多地限电停产，能耗双控政策持续加码 -> down
  [tc_007] 国务院金融稳定发展委员会召开专题会议 -> up
  [tc_008] 财政部、税务总局宣布印花税减半征收 -> up
  [tc_009] 国家数据局等17部门联合印发数据要素发展意见 -> neutral
  [tc_010] 工信部召开互联网行业专项整治行动会议 -> down


## 2. 自动生成反事实变体

对每条新闻生成两种反事实变体：
- **reverse_outcome**: 反转核心结论方向（测试模型是否依赖记忆而非文本分析）
- **alter_numbers**: 大幅修改数值至可能改变结论方向（测试模型对数量信息的敏感度）

共生成 42×2 = 84 个反事实变体。

In [3]:
client = LLMClient()

# 并发生成所有反事实变体 (42 cases × 2 types = 84 个任务，线程池并发)
print(f"并发生成 {len(test_cases) * len(list(VariantType))} 个反事实变体 ...")
all_variants = generate_counterfactuals_batch(client, test_cases, validate=True)

# 按 case 分组打印结果
for tc in test_cases:
    case_variants = [v for v in all_variants if v.original_case_id == tc.id]
    print(f"[{tc.id}] {tc.news.title[:30]}...")
    for v in case_variants:
        status = v.modification_description or "ok"
        print(f"  {v.variant_type}: {status}")

print(f"\nTotal variants: {len(all_variants)}")
save_counterfactual_variants(all_variants)
print("Saved to data/generated/counterfactual_variants.json")

并发生成 84 个反事实变体 ...


[tc_001] 央行宣布降准降息，推出多项金融支持政策...
  VariantType.REVERSE_OUTCOME: validated (attempt 1)
  VariantType.ALTER_NUMBERS: validated (attempt 1)
[tc_002] A股三大指数大幅低开，沪指跌破2700点...
  VariantType.REVERSE_OUTCOME: validated (attempt 1)
  VariantType.ALTER_NUMBERS: validated (attempt 1)
[tc_003] 中央政治局会议定调下半年经济工作...
  VariantType.REVERSE_OUTCOME: validated (attempt 1)
  VariantType.ALTER_NUMBERS: validated (attempt 1)
[tc_004] 中办国办印发意见：进一步减轻义务教育阶段学生作业负担和校外培...
  VariantType.REVERSE_OUTCOME: max retries reached (2)
  VariantType.ALTER_NUMBERS: validated (attempt 1)
[tc_005] 证监会主席吴清首次公开亮相，强调从严监管...
  VariantType.REVERSE_OUTCOME: validated (attempt 1)
  VariantType.ALTER_NUMBERS: validated (attempt 1)
[tc_006] 多地限电停产，能耗双控政策持续加码...
  VariantType.REVERSE_OUTCOME: validated (attempt 1)
  VariantType.ALTER_NUMBERS: validated (attempt 1)
[tc_007] 国务院金融稳定发展委员会召开专题会议...
  VariantType.REVERSE_OUTCOME: validated (attempt 1)
  VariantType.ALTER_NUMBERS: validated (attempt 1)
[tc_008] 财政部、税务总局宣布印花税减半征收...
  Variant

## 3. 预览反事实变体

In [4]:
import random

# Side-by-side comparison for each variant type (random sample, rerun to refresh)
tc_map = {tc.id: tc for tc in test_cases}
for vt in VariantType:
    candidates = [v for v in all_variants if v.variant_type == vt.value]
    if candidates:
        example = random.choice(candidates)
        orig_tc = tc_map.get(example.original_case_id)
        orig_text = orig_tc.news.content if orig_tc else "(original not found)"
        show_comparison(orig_text, example.modified_content, title=f"变体类型: {vt.value} ({example.original_case_id})")

原文,修改后
2022年6月16日，美联储宣布加息75个基点，将联邦基金利率目标区间上调至1.50%-1.75%，为1994年以来最大单次加息幅度。美联储主席鲍威尔表示不排除7月继续加息75个基点的可能性。全球资本市场剧烈波动。,2022年6月16日，美联储宣布降息75个基点，将联邦基金利率目标区间下调至1.50%-1.75%，为1994年以来最大单次降息幅度。美联储主席鲍威尔表示不排除7月继续降息75个基点的可能性。全球资本市场趋于稳定。


原文,修改后
2023年8月27日，财政部、税务总局宣布自8月28日起证券交易印花税实施减半征收。同日证监会宣布阶段性收紧IPO节奏，规范大股东减持行为。三大利好政策同时出台，被市场称为'组合拳'。,2023年8月27日，财政部、税务总局宣布自8月28日起证券交易印花税实施小幅上调。同日证监会宣布阶段性放宽IPO节奏，适度放开大股东减持行为。三项政策调整同时出台，被市场称为'组合拳'。


## 4. 生成掩码版本预览

展示各维度掩码效果，包括 rule-based 和 LLM-based 两种模式。

In [5]:
from src.masking import mask_year, mask_entity, mask_numbers, mask_sector, apply_masking, MaskingConfig

# Random sample, rerun to refresh
sample = random.choice(test_cases)
text = sample.news.content
print(f"样本: [{sample.id}] {sample.news.title}")

# Rule-based masking comparisons
show_comparison(text, mask_year(text), title="年份掩码 (rule-based)")
show_comparison(text, mask_entity(text, sample.key_entities), title="实体掩码 (rule-based)")
show_comparison(text, mask_numbers(text), title="数字掩码 (rule-based)")
show_comparison(text, mask_sector(text), title="板块掩码 (rule-based)")

full_config = MaskingConfig(mask_year=True, mask_entity=True, mask_numbers=True, mask_sector=True)
show_comparison(text, apply_masking(text, full_config, sample.key_entities, client=client), title="全掩码 (rule-based)")

# LLM-based masking comparisons
llm_config_year = MaskingConfig(mask_year=True, mask_mode="llm")
llm_config_entity = MaskingConfig(mask_entity=True, mask_mode="llm")
llm_config_numbers = MaskingConfig(mask_numbers=True, mask_mode="llm")
llm_config_sector = MaskingConfig(mask_sector=True, mask_mode="llm")
llm_config_full = MaskingConfig(mask_year=True, mask_entity=True, mask_numbers=True, mask_sector=True, mask_mode="llm")

show_comparison(text, apply_masking(text, llm_config_year, sample.key_entities, client=client), title="年份掩码 (LLM-based)")
show_comparison(text, apply_masking(text, llm_config_entity, sample.key_entities, client=client), title="实体掩码 (LLM-based)")
show_comparison(text, apply_masking(text, llm_config_numbers, sample.key_entities, client=client), title="数字掩码 (LLM-based)")
show_comparison(text, apply_masking(text, llm_config_sector, sample.key_entities, client=client), title="板块掩码 (LLM-based)")
show_comparison(text, apply_masking(text, llm_config_full, sample.key_entities, client=client), title="全掩码 (LLM-based)")

样本: [tc_002] A股三大指数大幅低开，沪指跌破2700点


原文,修改后
2024年2月5日，A股三大指数集体大幅低开。沪指跌破2700点关口，创近5年新低。深成指跌超3%，创业板指跌超4%。两市超4800只个股下跌，市场恐慌情绪蔓延。融资盘强制平仓和量化基金集中卖出被认为是主要推手。,[YEAR]年2月5日，A股三大指数集体大幅低开。沪指跌破2700点关口，创近5年新低。深成指跌超3%，创业板指跌超4%。两市超4800只个股下跌，市场恐慌情绪蔓延。融资盘强制平仓和量化基金集中卖出被认为是主要推手。


原文,修改后
2024年2月5日，A股三大指数集体大幅低开。沪指跌破2700点关口，创近5年新低。深成指跌超3%，创业板指跌超4%。两市超4800只个股下跌，市场恐慌情绪蔓延。融资盘强制平仓和量化基金集中卖出被认为是主要推手。,2024年2月5日，A股三大指数集体大幅低开。[实体1]跌破2700点关口，创近5年新低。[实体2]跌超3%，[实体3]跌超4%。两市超4800只个股下跌，市场恐慌情绪蔓延。融资盘强制平仓和量化基金集中卖出被认为是主要推手。


原文,修改后
2024年2月5日，A股三大指数集体大幅低开。沪指跌破2700点关口，创近5年新低。深成指跌超3%，创业板指跌超4%。两市超4800只个股下跌，市场恐慌情绪蔓延。融资盘强制平仓和量化基金集中卖出被认为是主要推手。,2024年2月5日，A股三大指数集体大幅低开。沪指跌破2700点关口，创近5年新低。深成指跌超[NUM]%，创业板指跌超[NUM]%。两市超4800只个股下跌，市场恐慌情绪蔓延。融资盘强制平仓和量化基金集中卖出被认为是主要推手。


原文,修改后
2024年2月5日，A股三大指数集体大幅低开。沪指跌破2700点关口，创近5年新低。深成指跌超3%，创业板指跌超4%。两市超4800只个股下跌，市场恐慌情绪蔓延。融资盘强制平仓和量化基金集中卖出被认为是主要推手。,2024年2月5日，A股三大指数集体大幅低开。沪指跌破2700点关口，创近5年新低。深成指跌超3%，创业板指跌超4%。两市超4800只个股下跌，市场恐慌情绪蔓延。融资盘强制平仓和量化基金集中卖出被认为是主要推手。


...

原文,修改后
2024年2月5日，A股三大指数集体大幅低开。沪指跌破2700点关口，创近5年新低。深成指跌超3%，创业板指跌超4%。两市超4800只个股下跌，市场恐慌情绪蔓延。融资盘强制平仓和量化基金集中卖出被认为是主要推手。,近期，A股三大指数集体大幅低开。沪指跌破2700点关口，创下近年新低。深成指跌超3%，创业板指跌超4%。两市超4800只个股下跌，市场恐慌情绪蔓延。融资盘强制平仓和量化基金集中卖出被认为是主要推手。


原文,修改后
2024年2月5日，A股三大指数集体大幅低开。沪指跌破2700点关口，创近5年新低。深成指跌超3%，创业板指跌超4%。两市超4800只个股下跌，市场恐慌情绪蔓延。融资盘强制平仓和量化基金集中卖出被认为是主要推手。,2024年2月5日，A股三大指数集体大幅低开。沪指跌破2700点关口，创近5年新低。深成指跌超3%，创业板指跌超4%。两市超4800只个股下跌，市场恐慌情绪蔓延。融资盘强制平仓和部分机构投资者的集中卖出被认为是主要推手。


原文,修改后
2024年2月5日，A股三大指数集体大幅低开。沪指跌破2700点关口，创近5年新低。深成指跌超3%，创业板指跌超4%。两市超4800只个股下跌，市场恐慌情绪蔓延。融资盘强制平仓和量化基金集中卖出被认为是主要推手。,2024年2月5日，A股三大指数集体大幅低开。沪指跌破关键整数关口，创下近年新低。深成指与创业板指均出现显著下跌，跌幅分别超过一定幅度。两市绝大多数个股收跌，市场恐慌情绪有所蔓延。融资盘强制平仓与量化基金集中卖出被认为是市场承压的主要推手。


原文,修改后
2024年2月5日，A股三大指数集体大幅低开。沪指跌破2700点关口，创近5年新低。深成指跌超3%，创业板指跌超4%。两市超4800只个股下跌，市场恐慌情绪蔓延。融资盘强制平仓和量化基金集中卖出被认为是主要推手。,2024年2月5日，A股三大指数集体大幅低开。沪指跌破2700点关口，创近5年新低。深成指跌超3%，创业板指跌超4%。两市超4800只个股下跌，市场恐慌情绪蔓延。杠杆资金强制平仓和部分机构资金的集中卖出被认为是主要推手。


原文,修改后
2024年2月5日，A股三大指数集体大幅低开。沪指跌破2700点关口，创近5年新低。深成指跌超3%，创业板指跌超4%。两市超4800只个股下跌，市场恐慌情绪蔓延。融资盘强制平仓和量化基金集中卖出被认为是主要推手。,近期，A股三大指数集体大幅低开。沪指跌破一个重要整数关口，创下近年新低。深成指与创业板指均出现显著下跌。两市绝大多数个股下跌，市场恐慌情绪蔓延。融资盘强制平仓和某类基金的集中卖出被认为是主要推手。


## 5. 数据统计

In [6]:
import pandas as pd

stats = []
for tc in test_cases:
    stats.append({
        "id": tc.id,
        "category": tc.news.category.value,
        "year": tc.news.publish_time.year,
        "direction": tc.expected_direction.value,
        "sector": tc.sector,
        "n_entities": len(tc.key_entities),
        "n_numbers": len(tc.key_numbers),
    })

df = pd.DataFrame(stats)
print("测试用例分布：")
print(f"\n按类别：\n{df['category'].value_counts()}")
print(f"\n按年份：\n{df['year'].value_counts().sort_index()}")
print(f"\n按方向：\n{df['direction'].value_counts()}")
print(f"\n按板块：\n{df['sector'].value_counts()}")

测试用例分布：

按类别：
category
corporate    16
industry     12
policy       10
macro         4
Name: count, dtype: int64

按年份：
year
2020     8
2021     8
2022     8
2023    10
2024     8
Name: count, dtype: int64

按方向：
direction
up         19
down       17
neutral     6
Name: count, dtype: int64

按板块：
sector
大盘     8
消费     4
半导体    4
科技     3
互联网    3
金融     2
新能源    2
医药     2
房地产    2
教育     1
煤炭     1
农业     1
电力     1
制造     1
汽车     1
CXO    1
光伏     1
白酒     1
军工     1
锂电     1
银行     1
Name: count, dtype: int64
